# Profiling 实操：采集基线与融合 trace

为了快速获取与度量融合算子的收益，本节让两次小规模等效运行在相同窗口采集 rank 0 的 Step 5。
---

## Profiling 配置说明

Wordle recipe 使用 `TrainerConfig` 中的默认 `ProfilingConfig`。Baseline recipe 执行原生 RMSNorm 路径，Fused recipe 应加载 `npu_rms_norm` converter。与本节相关的默认值如下：


```python
profiling=ProfilingConfig(
    enable_profiling=False,      # 默认关闭
    enable_online_parse=True,    # 训练结束后解析 trace
    profile_ranks=[-1],          # profile 所有 rank
    profile_step_start=0,        # 未指定固定起始 step
    profile_step_end=0,          # 未指定固定结束 step
    profile_record_shapes=True,  # 记录张量形状
    profile_with_memory=False,   # 默认不记录显存
)
```


本节通过命令行覆盖四项配置：`profile_ranks=0` 只采集 rank 0；`profile_step_start=5`、`profile_step_end=6` 采集 Step 5；`profile_with_memory=True` 为显存分析生成 `memory_record.csv`。`profile_record_shapes=True` 已是默认值，训练命令直接沿用。

选择 rank 0 可以减少 profiling 开销，并让 `baseline`、`fused_op` 目录各产生一份 trace。Step 5 位于模型和 NPU 算子初始化之后，两次运行使用相同采集窗口。

### Profiling 的 batch size 设置

第 3 章训练用 `global_batch_size=64` 追求收敛质量。Profiling 时改为 `gbs=4`：


```text
gbs=64 → local batch 2、DP degree 2 时累计 16 个 microbatch
gbs=4  → local batch 2、DP degree 2 时累计 1 个 microbatch
```


每个 microbatch 中，每个 rank 处理 2 条 packed sequence，两个 data-parallel rank 合计处理 $2 \times 2=4$ 条。因此：

$$N_{acc}=\frac{B_{global}}{B_{local} \times D_{DP}}$$

其中 $N_{acc}$ 是 gradient accumulation steps，$B_{global}$ 是 global batch size，$B_{local}$ 是每个 rank 的 local batch size，$D_{DP}$ 是 data-parallel degree。

在本节固定 `local_batch_size=2` 和 `seq_len=1024` 时，把 GBS 从 64 调整为 4，会将一个 optimizer step 内累计的 microbatch 数从 16 降为 1。每个 rank 的模型输入仍为 `(2, 1024, ...)`，RMSNorm 接收的 shape、forward/backward 计算路径以及单次算子调用规模保持不变。因此，本节的单 microbatch 算子性能分析仍在相同输入规模下进行。

Baseline 与 Fused 都使用 `gbs=4`，两份 trace 覆盖相同的一次 microbatch 和一次参数更新，可以直接比较 RMSNorm launch 数、单次 device time 和 Step 5 的相对变化。Step 5 的绝对耗时对应这套 profiling 配置；`gbs=64` 的正式训练会在一次参数更新前执行 16 个 microbatch。

训练命令通过 `--training.global-batch-size 4` 设置该值。

---

## 训练命令

### Baseline：原生 RMSNorm



启动训练 + profiling：


In [ ]:
import os
original_dir = os.getcwd()
%cd /mnt/workspace/torchtitan-npu
%env BASH_ENV=/home/developer/Ascend/cann/set_env.sh

In [ ]:
%%bash
set -euo pipefail
# 清理上次运行的 baseline profiling
rm -rf outputs/checkpoints/baseline outputs/profile_traces/baseline
NGPU=2 \
DATASET_PATH=./assets/data/wordle \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh \
  --training.steps 10 \
  --training.global-batch-size 4 \
  --checkpoint.folder checkpoints/baseline \
  --profiling.enable-profiling \
  --profiling.profile-ranks 0 \
  --profiling.profile-step-start 5 \
  --profiling.profile-step-end 6 \
  --profiling.profile-with-memory \
  --profiling.save-traces-folder profile_traces/baseline \
dataloader:chat_data_loader_config \
  --dataloader.dataset_path "assets/data/wordle" \



### Fused：`npu_rms_norm` converter

回顾 04.02 ，在`torchtitan-npu`的 qwen3 配置：`torchtitan-npu/torchtitan_npu/models/qwen3/config_registry.py` 下修改源码，启用融合算子配置。

将

```python
def _qwen3_1_7b_converters() -> ModelConvertersContainer.Config:
    return ModelConvertersContainer.Config(converters=[])
```

修改为

```python
def _qwen3_1_7b_converters() -> ModelConvertersContainer.Config:
    return ModelConvertersContainer.Config(converters=[
        get_model_converter_config("npu_rms_norm")
    ])
```

后，用以下命令重新运行 sft_qwen3_1_7b_wordle 配置，以获得包含融合算子的配置。


In [ ]:
%%bash
set -euo pipefail
# 清理上次运行的 fused_op profiling
rm -rf outputs/checkpoints/fused_op outputs/profile_traces/fused_op
NGPU=2 \
DATASET_PATH=./assets/data/wordle \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh \
  --training.steps 10 \
  --training.global-batch-size 4 \
  --checkpoint.folder checkpoints/fused_op \
  --profiling.enable-profiling \
  --profiling.profile-ranks 0 \
  --profiling.profile-step-start 5 \
  --profiling.profile-step-end 6 \
  --profiling.profile-with-memory \
  --profiling.save-traces-folder profile_traces/fused_op \
dataloader:chat_data_loader_config \
  --dataloader.dataset_path "assets/data/wordle" \

两次训练应使用相同的模型、数据、batch 和采集窗口；训练计算路径的预期差异是 `model_converters`：

| 配置 | model_converters | 效果 |
|------|-----------------|------|
| baseline | `[]`（空） | 原生 PyTorch RMSNorm 路径 |
| fused | `["npu_rms_norm"]` | NPU RMSNorm converter 路径 |

训练命令的 `--profiling.save-traces-folder` 参数分别为 `profile_traces/baseline` 和 `profile_traces/fused_op`。实际 trace 输出目录是 `outputs/profile_traces/baseline` 和 `outputs/profile_traces/fused_op`。checkpoint 实际写入 `outputs/checkpoints/baseline` 和 `outputs/checkpoints/fused_op`。

---

两次训练完成后，进入 04.04 运行分析脚本并查看结果。


## 练习

1. （单选题）要采集 Step 5，本节应设置哪一组 profiling 区间？
    A. `profile_step_start=5`、`profile_step_end=6`
    B. `profile_step_start=0`、`profile_step_end=0`
    C. `profile_step_start=5`、`profile_step_end=5`
    D. `profile_step_start=6`、`profile_step_end=7`

2. （单选题）Profiling 时把 `global_batch_size` 改成 4 的主要目的是什么？
    A. 把 gradient accumulation 从 16 降为 1，同时保持每个 rank 的 microbatch shape 不变
    B. 自动提高模型精度
    C. 关闭梯度计算
    D. 让模型切换到 TP=4

3. （单选题）本轮两份 profiling 输出如何保持对应关系？
    A. 分别写入固定的 `profile_traces/baseline` 与 `profile_traces/fused_op` 目录
    B. 按文件修改时间决定角色
    C. 从统一目录随机选择两份文件
    D. 用 loss 数值推断角色

In [ ]:
%cd $original_dir
!cat ./answer/04.03_answer.txt